# SnowBallSLR in Colab

Automated bidirectional citation searching that stops on a stated criterion, estimates how much of the literature it found, and replays from its own cache.

Run the cells in order. The crawl is network-bound, so the free tier is enough — no GPU needed.

Repository: https://github.com/s-matysik/SnowBallSLR

## 1. Install

In [ ]:
!pip install -q git+https://github.com/s-matysik/SnowBallSLR.git

In [ ]:
import snowballslr
print('version:', snowballslr.__version__)

## 2. Configure the run

Two things matter more than the rest.

**Arms must be able to capture the same record.** A citation graph is temporally acyclic, so backward and forward expansion sample near-disjoint layers: arms defined by *direction* overlap on nothing and produce no recall estimate at all. Define them by *source* instead, as below.

**Set a screening budget.** Iteration 2 is typically 5-20x iteration 1. Measure your corpus on one iteration before committing.

In [ ]:
from pathlib import Path
from snowballslr.config import Config
from snowballslr.core.run import Run

MAILTO = 'you@example.org'   # both APIs are free; an address raises your rate limit
SEED_DOIS = ['10.1145/2601248.2601268']   # replace with your own seed set

config = Config.from_dict({
    'run': {'name': 'colab-demo', 'max_iterations': 2, 'directions': ['backward']},
    'providers': {
        'order': ['openalex', 'crossref'],
        'openalex': {'enabled': True, 'rps': 4, 'mailto': MAILTO},
        'crossref': {'enabled': True, 'rps': 4, 'mailto': MAILTO},
    },
    'eligibility': {'types': ['article', 'book-chapter'], 'require_title': True},
    'stopping': {'mode': 'any_of', 'rules': [
        {'type': 'budget', 'max_screened': 60},
        {'type': 'exhaustion'},
    ]},
    'estimate': {
        'arms': [
            {'name': 'openalex', 'filter': {'provider': 'openalex'}},
            {'name': 'crossref', 'filter': {'provider': 'crossref'}},
        ],
        'method': 'chapman',
    },
})

# Read these rather than skipping them: they catch designs that cannot work.
for warning in config.warnings():
    print('config warning:', warning)

## 3. Resolve seeds and expand one iteration

In [ ]:
run = Run.init(Path('run_demo'), seeds=SEED_DOIS, config=config)
print('seeds resolved:', len(run.state.works), 'of', len(SEED_DOIS))

candidates = run.step()
print('candidates to screen:', len(candidates))
for work in candidates[:5]:
    print(' ', work.year, (work.title or '')[:70])

## 4. Screen

Nothing decides relevance for you. For a handful of records, screen inline. For more, export a CSV, screen it outside the notebook, and read it back with `run.label_from_file('labels_001.csv')` — columns `key,decision[,note]`.

In [ ]:
import csv

# Export for screening elsewhere (recommended beyond a few dozen records).
with open('to_screen.csv', 'w', newline='', encoding='utf-8') as fh:
    writer = csv.writer(fh)
    writer.writerow(['key', 'decision', 'year', 'title', 'abstract'])
    for work in candidates:
        writer.writerow([work.key, '', work.year or '', work.title or '',
                         (work.abstract or '')[:1500]])
print('wrote to_screen.csv —', len(candidates), 'rows; fill the decision column')

In [ ]:
# Or screen inline. Enter i / e / s (include / exclude / skip).
decisions = {}
for work in candidates[:10]:
    print('\n', work.year, work.title)
    print((work.abstract or '[no abstract]')[:400])
    answer = input('include / exclude / skip? ').strip().lower()[:1]
    decisions[work.key] = {'i': 'include', 'e': 'exclude'}.get(answer, 'unscreened')

run.label(decisions)
print('phase:', run.state.phase, '| stopped by:', run.state.stopped_by)

## 5. Iterate until a rule fires

Repeat step-then-label until `run.state.phase` is `stopped`. Whichever rule fired is in `run.state.stopped_by`; its reasoning, written for a methods section, is in `run.state.last_decisions`.

In [ ]:
if run.state.phase != 'stopped':
    candidates = run.step()
    print('next iteration:', len(candidates), 'candidates')

for decision in (run.state.last_decisions or []):
    fired = '*** FIRED ***' if decision.get('triggered') else '             '
    print(fired, decision.get('rule'), '->', (decision.get('rationale') or '')[:120])

## 6. PRISMA outputs

In [ ]:
run.report()
!ls run_demo/outputs/

In [ ]:
from IPython.display import SVG, display
display(SVG('run_demo/outputs/prisma.svg'))

## 7. Keep the run — the Colab runtime is deleted when you close it

A completed run replays from its own cache with no API access, so archiving the run directory preserves the review exactly.

In [ ]:
!snowballslr verify run_demo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r run_demo /content/drive/MyDrive/
print('archived; reload later with Run.load("/content/drive/MyDrive/run_demo")')

## 8. Reproduce the paper's validation numbers

Offline and deterministic, but it lives in the repository rather than the package, so clone. `--quick` takes a couple of minutes; without it, the full 270-cell grid.

In [ ]:
!git clone -q https://github.com/s-matysik/SnowBallSLR.git
%cd SnowBallSLR
!pip install -q -e .
!python validation/reproduce.py --quick